In [ ]:
# ============================================================
# CELL 1 — Install packages
# This cell installs all the Python libraries we need:
#   - The project repo (semantic-correspondence)
#   - OpenCV for image processing
#   - tqdm for progress bars
#   - SAM (Segment Anything Model) from Meta
# You only need to run this once per Colab session.
# ============================================================

!test -d /content/semantic-correspondence || git clone -q https://github.com/aexomir/semantic-correspondence.git /content/semantic-correspondence
!pip -q install -U pip
!pip -q install -r /content/semantic-correspondence/requirements.txt
!pip -q install opencv-python tqdm
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# CELL 2 — Download DINO model code
# DINOv2 and DINOv3 are not on PyPI, so we clone their GitHub
# repos directly into this Colab environment.
# We also install the extra packages these repos require.
# ============================================================

!rm -rf dinov2 dinov3
!git clone -q https://github.com/facebookresearch/dinov2.git
!git clone -q https://github.com/facebookresearch/dinov3.git

!pip -q install omegaconf torchmetrics fvcore iopath submitit ftfy regex scikit-learn termcolor

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive and set file paths
# We store model weights and the dataset on Google Drive so
# they survive between sessions. This cell:
#   1. Imports all libraries used throughout the notebook
#   2. Connects to Google Drive
#   3. Defines paths to all model weight files (.pth)
#   4. Defines paths to the dataset files
#   5. Creates a local /content/data folder for fast disk access
# ============================================================

import os
import sys
import shutil
import types
import math
import json
import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from contextlib import nullcontext
import pandas as pd
from google.colab import drive

# Connect to Google Drive (will ask for permission the first time)
drive.mount('/content/drive', force_remount=True)

MYDRIVE = '/content/drive/MyDrive'

# Paths to saved model weights on Drive
DINOV2_WEIGHTS = os.path.join(MYDRIVE, 'dinov2_vitl14_pretrain_big.pth')
DINOV3_WEIGHTS = os.path.join(MYDRIVE, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
SAM_WEIGHTS_B  = os.path.join(MYDRIVE, 'sam_vit_l_0b3195_big.pth')

# Paths to dataset files on Drive
SPAIR_DIR = os.path.join(MYDRIVE, 'SPair-71k')
SPAIR_TAR = os.path.join(MYDRIVE, 'SPair-71k.tar.gz')
PFW_ZIP   = os.path.join(MYDRIVE, 'pf-willow.zip')

# Local fast disk folder (on Colab's machine, not Drive)
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.isdir(MYDRIVE):
    raise RuntimeError('Drive not mounted. Complete the auth prompt and re-run this cell.')

print('OK:', MYDRIVE)


Mounted at /content/drive
OK: /content/drive/MyDrive


In [ ]:
# ============================================================
# CELL 4 — Copy the SPair-71k dataset to local disk
# Extract the tar.gz file directly into the local data directory,
# or copy the directory if the tar file is missing.
# ============================================================
import shutil

local_spair = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')

if not os.path.exists(local_spair):
    if os.path.exists(SPAIR_TAR):
        print(f"Extracting {SPAIR_TAR} to {LOCAL_DATA_DIR}...")
        !tar -xzf {SPAIR_TAR} -C {LOCAL_DATA_DIR}
        print("Extraction complete!")
    elif os.path.exists(SPAIR_DIR):
        print(f"tar file not found. Copying directory {SPAIR_DIR} to {local_spair}...")
        shutil.copytree(SPAIR_DIR, local_spair)
        print("Copy complete!")
    else:
        print("Error: Neither tar file nor directory found in Drive.")
else:
    print(f"Dataset already exists at {local_spair}")

Extracting /content/drive/MyDrive/SPair-71k.tar.gz to /content/data...
Extraction complete!


In [ ]:
# ============================================================
# CELL 5 — Load the three pretrained models
# We load and compare three foundation models:
#   - DINOv2 ViT-L/14: self-supervised ViT from Meta (large)
#   - DINOv3 ViT-B/16: updated version of DINOv2
#   - SAM ViT-L: Segment Anything Model encoder from Meta (large)
# For each model we build the architecture, load saved weights
# from Google Drive, then move to GPU and set to eval mode.
# ============================================================

from segment_anything import SamPredictor, sam_model_registry

# Load SAM ViT-L
sam = sam_model_registry['vit_l'](checkpoint=SAM_WEIGHTS_B)
sam_predictor = SamPredictor(sam)

# Load DINOv2 ViT-L/14 (big weight file uses ViT-L architecture)
dinov2 = torch.hub.load('dinov2', 'dinov2_vitl14', source='local', weights=DINOV2_WEIGHTS)

# Load DINOv3 ViT-B/16 (small file — big file uses incompatible architecture)
dinov3 = torch.hub.load('dinov3', 'dinov3_vitb16', source='local', weights=DINOV3_WEIGHTS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam.to(device).eval()
dinov2.to(device).eval()
dinov3.to(device).eval()

print('Ready. Device:', device)


/content/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "file:///content/drive/MyDrive/dinov2_vitl14_pretrain_big.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain_big.pth


100%|██████████| 1.13G/1.13G [00:35<00:00, 34.6MB/s]


Downloading: "file:///content/drive/MyDrive/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth


100%|██████████| 327M/327M [00:12<00:00, 28.3MB/s]


Ready. Device: cuda


In [ ]:
# ============================================================
# CELL 6 — Feature extraction functions
# These three functions run an image through a model and
# return its internal feature map as a torch tensor.
# A 'feature map' is a grid of vectors — one vector per image
# patch — that describes what the model 'sees' in that region.
# ============================================================


def extract_dino_features(model, img_tensor, trainable=False):
    """
    Run an image through a DINO model and return its patch features.

    The model splits the image into a grid of small patches and produces
    one feature vector per patch. We reshape these vectors into a 2D
    spatial grid so they are easy to work with.

    Input:  img_tensor — a torch tensor of shape (1, C, H, W)
            trainable  — if True, keeps gradients (for finetuning);
                         if False, runs under no_grad and forces eval mode
    Output: torch tensor of shape (1, D, grid_H, grid_W)
              D = feature dimension, grid_H/W = number of patches per side
    """
    if not trainable:
        model.eval()

    ctx = torch.no_grad() if not trainable else nullcontext()
    with ctx:
        out = model.forward_features(img_tensor)
        # Different DINO versions store patch tokens under different key names
        if isinstance(out, dict):
            patch_tokens = out.get('x_norm_patchtokens', out.get('x_norm_patch_tokens'))
        else:
            patch_tokens = out

        if patch_tokens is None:
            keys = list(out.keys()) if isinstance(out, dict) else 'N/A'
            raise ValueError(f'Could not find patch tokens. Keys: {keys}')

        b, n, d = patch_tokens.shape
        grid = int(math.isqrt(n))  # num_patches must be a perfect square (e.g. 37*37=1369)
        if grid * grid != n:
            raise ValueError(f'Expected square number of patches, got N={n}')

        # Rearrange from (batch, num_patches, dim) -> (batch, dim, grid_H, grid_W)
        return patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)


def extract_dino_layers(model, img_tensor, layer_ids):
    """
    Run an image through DINO and return features from specific middle layers.

    Instead of just the final output, we can inspect features at any transformer
    block. This is useful when fine-tuning — you want to unfreeze only the last
    few layers and see which layers help the most.

    Input:  img_tensor — torch tensor (1, C, H, W)
            layer_ids  — list of layer indices to extract, e.g. [10, 11]
    Output: dict mapping each layer_id -> torch tensor (1, D, grid_H, grid_W)
    """
    model.eval()
    with torch.no_grad():
        # Get the output of every transformer block at once
        all_layers = model.get_intermediate_layers(img_tensor, n=len(model.blocks), norm=True)
        out = {}
        for lid in layer_ids:
            patch_tokens = all_layers[lid]
            b, n, d = patch_tokens.shape
            grid = int(math.isqrt(n))
            if grid * grid != n:
                raise ValueError(f'Layer {lid}: expected square patches, got N={n}')
            # Rearrange to (batch, dim, grid_H, grid_W)
            out[lid] = patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)
        return out


def extract_sam_features(predictor, image_pil, res=1024):
    """Run an image through SAM's image encoder and return its feature map.

    To match SAM's pretrained setup (and avoid any positional-embedding
    interpolation), we run SAM at its native resolution: 1024×1024.

    Input:  image_pil — RGB PIL Image
            res       — must be 1024
    Output: torch tensor of shape (1, C, feat_H, feat_W)
    """
    if int(res) != 1024:
        raise ValueError(f"SAM without pos-embed resizing requires res=1024, got {res}")

    device = next(predictor.model.parameters()).device
    encoder = predictor.model.image_encoder

    # Resize image to 1024×1024
    image_resized = transforms.functional.resize(
        image_pil, (1024, 1024), interpolation=transforms.InterpolationMode.BILINEAR
    )

    # Convert to torch tensor (values in [0, 255]) and normalize using SAM's mean/std
    img_tensor = transforms.ToTensor()(image_resized).unsqueeze(0).to(device) * 255.0
    pixel_mean = torch.tensor([123.675, 116.28, 103.53], device=device).view(-1, 1, 1)
    pixel_std  = torch.tensor([58.395, 57.12, 57.375],  device=device).view(-1, 1, 1)
    img_tensor = (img_tensor - pixel_mean) / pixel_std

    with torch.no_grad():
        feats = encoder(img_tensor)

    return feats


In [ ]:
# ============================================================
# CELL 7 — Dataset paths, model registry, and feature cache
# This cell does three things:
#   1. Defines where all data and results live on disk
#   2. Lists all three models we want to test (MODEL_SPECS)
#   3. Defines helper functions to save/load feature files
#      as .pt (PyTorch) files so we never re-run models on
#      the same image twice
# ============================================================

# --- Dataset and output folder paths ---
SPAIR_ROOT = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')
JPEG_ROOT = os.path.join(SPAIR_ROOT, 'JPEGImages')              # folder with all the images
PAIRANN_TEST_ROOT = os.path.join(SPAIR_ROOT, 'PairAnnotation', 'test')  # folder with keypoint labels

FEATURE_ROOT = os.path.join(MYDRIVE, 'features', 'spair71k')   # where we save computed features
RESULTS_ROOT = os.path.join(MYDRIVE, 'results')                 # where we save evaluation results

# --- Model registry ---
# Each entry says: which model object to use, what image size it needs, and what type it is.
# 'dino' models use the torchvision preprocessing pipeline.
# 'sam' models use SAM's own preprocessing inside extract_sam_features.
MODEL_SPECS = {
    'dinov2_vitl14': {
        'model': dinov2,
        'img_size': (518, 518),   # 518 / 14 approx 37 patches per side
        'kind': 'dino',
    },
    'dinov3_vitb16': {
        'model': dinov3,
        'img_size': (512, 512),   # 512 / 16 = 32 patches per side
        'kind': 'dino',
    },
    'sam_vit_l_res1024': {
        'model': sam_predictor,
        'img_size': (1024, 1024),
        'kind': 'sam',
        'sam_res': 1024,
    },
}


def _ensure_parent_dir(path):
    """Create the folder that contains the given file path, if it does not exist yet."""
    os.makedirs(os.path.dirname(path), exist_ok=True)


def _rel_from_jpeg_root(image_path):
    """Return the path of an image relative to the dataset images folder (JPEG_ROOT)."""
    rel = os.path.relpath(image_path, JPEG_ROOT)
    if rel.startswith('..'):
        raise ValueError(f'Expected image under JPEG_ROOT={JPEG_ROOT}, got {image_path}')
    return rel


def feature_path(model_key, image_path, layer_id=None):
    """Build the file path where we store cached features for one image.

    The path mirrors the dataset folder structure so features are easy to find.
    Example: features/spair71k/dinov2_vitl14/dog/image001.pt
    """
    rel = _rel_from_jpeg_root(image_path)
    rel_no_ext = os.path.splitext(rel)[0]
    # If layer_id is provided (DINO intermediate layer), cache it separately
    layer_tag = '' if layer_id is None else f'_layer{int(layer_id)}'
    return os.path.join(FEATURE_ROOT, model_key, rel_no_ext + layer_tag + '.pt')


def save_feature_pt(path, feat_tensor):
    """Save a feature tensor to a .pt file. Creates the parent folder if needed."""
    _ensure_parent_dir(path)
    torch.save(feat_tensor, path)


def load_feature_pt(path):
    """Load and return the feature tensor stored in a .pt file."""
    return torch.load(path, map_location='cpu')


def dino_transform(img_size):
    """
    Build a torchvision preprocessing pipeline for DINO models.

    Three steps:
      1. Resize the image to img_size using bicubic interpolation
      2. Convert pixel values from [0, 255] integers to [0.0, 1.0] floats (ToTensor)
      3. Normalize with ImageNet mean and std (what DINO was trained with)

    Returns a torchvision Compose object — call it like a function on a PIL image.
    """
    return transforms.Compose(
        [
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


# Pre-build one transform pipeline per DINO model so we don't recreate it each time
_DINO_TRANSFORMS = {
    k: dino_transform(v['img_size'])
    for k, v in MODEL_SPECS.items()
    if v['kind'] == 'dino'
}


def compute_feature(model_key, image_path, layer_id=None):
    """
    Load one image, run it through the specified model, and return its feature map
    as a float16 torch tensor stored on CPU.

    For DINO models:
      - Apply the torchvision preprocessing pipeline (_DINO_TRANSFORMS)
      - Add a batch dimension and move to GPU
      - Run extract_dino_features -> get a torch tensor (1, D, H, W)

    For SAM:
      - Open image with PIL, convert to raw array for cv2
      - Run extract_sam_features -> get a torch tensor (1, C, H, W)

    Returns a float16 CPU tensor (D, H, W).
    Storing as float16 halves the disk space compared to float32.
    """
    spec = MODEL_SPECS[model_key]
    kind = spec['kind']

    img_pil = Image.open(image_path).convert('RGB')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if kind == 'dino':
        t = _DINO_TRANSFORMS[model_key](img_pil).unsqueeze(0).to(device)  # (1, C, H, W)
        if layer_id is None:
            feat = extract_dino_features(spec['model'], t)                     # (1, D, H, W)
        else:
            feat = extract_dino_layers(spec['model'], t, [int(layer_id)])[int(layer_id)]  # (1, D, H, W)
    elif kind == 'sam':
        feat = extract_sam_features(spec['model'], img_pil, res=spec['sam_res'])  # (1, C, H, W)
    else:
        raise ValueError(f'Unknown kind: {kind}')

    # Remove batch dim, store as float16 on CPU to save memory
    return feat.squeeze(0).detach().half().cpu()


def load_or_compute_feature(model_key, image_path, layer_id=None):
    """
    Return the feature tensor for one image, using the disk cache when possible.

    - If a cached .pt file already exists on Drive -> load and return it immediately (fast)
    - If not -> run the model, save the result to Drive, then return it

    This means each image is processed by the model only once across all runs.
    """
    out_path = feature_path(model_key, image_path, layer_id=layer_id)
    if os.path.exists(out_path):
        return load_feature_pt(out_path)

    feat = compute_feature(model_key, image_path, layer_id=layer_id)
    save_feature_pt(out_path, feat)
    return feat


print('Cache root:', FEATURE_ROOT)
print('Results root:', RESULTS_ROOT)
print('JPEG root:', JPEG_ROOT)
print('PairAnnotation test root:', PAIRANN_TEST_ROOT)


Cache root: /content/drive/MyDrive/features/spair71k
Results root: /content/drive/MyDrive/results
JPEG root: /content/data/SPair-71k/JPEGImages
PairAnnotation test root: /content/data/SPair-71k/PairAnnotation/test


In [ ]:
# ============================================================
# CELL 8 — Pre-compute and cache features for all images
# Running a model on 1800 images every time we evaluate would
# be slow. Instead, we run each model once on all images and
# save the results as .pt (PyTorch) files on Google Drive.
# On future runs, we just load the saved files.
# ============================================================

# Only process files with these extensions
VALID_IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}


def iter_jpeg_images():
    """
    Walk through the dataset images folder and yield the full path
    of every valid image file (.jpg, .jpeg, .png).
    """
    for root, _, files in os.walk(JPEG_ROOT):
        for fn in files:
            ext = os.path.splitext(fn)[1].lower()
            if ext in VALID_IMAGE_EXTS:
                yield os.path.join(root, fn)


def precompute_features(model_keys=None, limit=None):
    """
    Run all images through each model and save their feature tensors to Drive.

    - model_keys: which models to run (default: all models in MODEL_SPECS)
    - limit: optionally process only the first N images (useful for quick tests)

    For each image, if the feature file already exists on Drive we skip it.
    This makes the function safe to re-run — it only processes new images.
    """
    if model_keys is None:
        model_keys = list(MODEL_SPECS.keys())

    images = list(iter_jpeg_images())
    if limit is not None:
        images = images[: int(limit)]

    print(f'Found {len(images)} images under {JPEG_ROOT}')
    print('Models:', model_keys)

    for model_key in model_keys:
        missing = 0
        for img_path in tqdm(images, desc=f'{model_key}: caching'):
            out_path = feature_path(model_key, img_path)
            if os.path.exists(out_path):
                continue  # already cached, skip

            feat = compute_feature(model_key, img_path)
            save_feature_pt(out_path, feat)
            missing += 1

        print(f'{model_key}: newly cached {missing} feature files')


# Example usage:
# precompute_features(['dinov2_vitb14'], limit=50)   # quick test on 50 images
# precompute_features(list(MODEL_SPECS.keys()))       # full run on all models


In [ ]:
# ============================================================
# CELL 9 — Evaluation functions (PCK metric)
# This cell defines the full evaluation pipeline.
# For each pair of images in the test set we:
#   1. Load the cached feature tensors for both images
#   2. For each labeled keypoint in the source image,
#      find the most similar patch in the target image
#   3. Check if the predicted location is close enough to
#      the ground-truth keypoint (measured by PCK)
# PCK = Percentage of Correct Keypoints
# A prediction counts as 'correct' if its distance from the
# ground truth is below a threshold x (object bounding box size)
# ============================================================

# We evaluate at four strictness levels: 5%, 10%, 15%, 20% of bbox size
PCK_THRESHOLDS = [0.05, 0.1, 0.15, 0.2]


def list_test_pair_files():
    """
    Find all JSON annotation files in the SPair-71k test split.
    Each JSON file describes one image pair and contains the
    source/target keypoints and bounding box info.
    """
    if not os.path.isdir(PAIRANN_TEST_ROOT):
        raise FileNotFoundError(f'Missing PairAnnotation test dir: {PAIRANN_TEST_ROOT}')

    pair_files = []
    for root, _, files in os.walk(PAIRANN_TEST_ROOT):
        for fn in files:
            if fn.endswith('.json'):
                pair_files.append(os.path.join(root, fn))

    pair_files.sort()
    return pair_files


def _parse_xy(p):
    """
    Safely read a keypoint coordinate from the annotation.
    Returns (x, y) as floats, or None if the point is missing,
    invalid, or has negative coordinates (means it was not labeled).
    """
    if p is None:
        return None
    if not isinstance(p, (list, tuple)):
        return None
    if len(p) < 2:
        return None
    x, y = float(p[0]), float(p[1])
    if x < 0 or y < 0:
        return None
    return x, y


def softargmax_2d(similarity_map, temperature=0.01, window_size=5):
    """
    Find the best-matching location in a similarity map with sub-pixel accuracy.

    Plain argmax snaps to the nearest grid cell. This function does better:
      1. Find the peak grid cell with argmax
      2. Cut out a small window (e.g. 5x5) around that peak
      3. Apply softmax to the window values to get a probability distribution
      4. Compute the weighted average position — this gives a fractional (x, y)

    The temperature parameter controls how 'sharp' the softmax is.
    A low temperature (e.g. 0.01) makes it almost the same as argmax.

    Args:
        similarity_map: 2D torch tensor (H, W) of similarity scores
        temperature:    softmax temperature (lower = sharper)
        window_size:    size of the window around the peak

    Returns (pred_x, pred_y) in feature-map coordinates (sub-pixel precision).
    """
    H, W = similarity_map.shape
    device = similarity_map.device

    # Step 1: find the peak cell
    flat_idx = similarity_map.argmax().item()
    peak_y = flat_idx // W
    peak_x = flat_idx % W

    # Step 2: define window around the peak (clamped to map boundaries)
    half = window_size // 2
    y_start = max(0, peak_y - half)
    y_end   = min(H, peak_y + half + 1)
    x_start = max(0, peak_x - half)
    x_end   = min(W, peak_x + half + 1)

    window = similarity_map[y_start:y_end, x_start:x_end]

    # Step 3: softmax — apply temperature scaling
    window_flat = window.reshape(-1)
    probs = F.softmax(window_flat / temperature, dim=0)

    # Step 4: build coordinate grids for the window
    y_coords = torch.arange(y_start, y_end, device=device).float()
    x_coords = torch.arange(x_start, x_end, device=device).float()
    grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')

    # Step 5: weighted average position
    pred_x = (probs * grid_x.reshape(-1)).sum()
    pred_y = (probs * grid_y.reshape(-1)).sum()
    return pred_x, pred_y


def evaluate_cached(
    model_key,
    use_softargmax,
    layer_id=None,
    softargmax_temp=0.01,
    softargmax_window=5,
    thresholds=PCK_THRESHOLDS,
    max_pairs=None,
    save_csv=True,
):
    """
    Main evaluation function. Measures how well a model does at
    semantic correspondence on the SPair-71k test set.

    For each test image pair:
      - Load cached feature tensors for source and target images
      - L2-normalize them with F.normalize
      - For each source keypoint, extract its feature vector
      - Compute cosine similarity with every patch in the target
      - Predict the match using argmax or soft-argmax
      - Check if the prediction is within threshold x bbox_size of the truth

    Parameters:
      model_key       — which model to evaluate (must be in MODEL_SPECS)
      use_softargmax  — if True, use window soft-argmax; if False, use plain argmax
      max_pairs       — limit to first N pairs (useful for quick tests)
      save_csv        — if True, save per-image results to a CSV on Drive

    Returns a dict with overall PCK scores and per-image results.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    pair_files = list_test_pair_files()
    if max_pairs is not None:
        pair_files = pair_files[: int(max_pairs)]

    correct_kps = {t: 0 for t in thresholds}  # count of correct predictions per threshold
    total_kps = 0                              # total keypoints seen

    per_image_results = []

    for pair_path in tqdm(pair_files, desc=f'Evaluating {model_key}'):
        # Load the annotation for this pair
        with open(pair_path, 'r') as f:
            ann = json.load(f)

        category = ann['category']
        src_name = ann['src_imname']
        trg_name = ann['trg_imname']

        src_img_path = os.path.join(JPEG_ROOT, category, src_name)
        trg_img_path = os.path.join(JPEG_ROOT, category, trg_name)

        # Get the pixel size of each image (needed to map feature coords back to pixels)
        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # Load (or compute) feature tensors, upcast to float32, L2-normalize, move to device
        f_src = F.normalize(load_or_compute_feature(model_key, src_img_path, layer_id=layer_id).float(), dim=0).to(device)
        f_trg = F.normalize(load_or_compute_feature(model_key, trg_img_path, layer_id=layer_id).float(), dim=0).to(device)

        fh, fw = f_src.shape[1], f_src.shape[2]   # feature grid height and width
        if f_src.shape[1:] != f_trg.shape[1:]:
            raise ValueError(f'Feature grids mismatch: src={f_src.shape[1:]} trg={f_trg.shape[1:]}')

        src_kps = ann['src_kps']
        trg_kps = ann['trg_kps']

        # PCK threshold is normalized by the largest side of the target bounding box
        trg_bbox = ann['trg_bndbox']
        bbox_w = float(trg_bbox[2] - trg_bbox[0])
        bbox_h = float(trg_bbox[3] - trg_bbox[1])
        norm_factor = max(bbox_w, bbox_h)

        image_correct = {t: 0 for t in thresholds}
        image_total_kps = 0

        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        for idx in kp_indices:
            p_src = _parse_xy(src_kps[idx])
            p_trg = _parse_xy(trg_kps[idx])
            if p_src is None or p_trg is None:
                continue  # skip unlabeled or invalid keypoints

            sx, sy = p_src   # source keypoint in pixel coordinates
            tx, ty = p_trg   # target ground-truth in pixel coordinates

            # Convert source keypoint from pixel coords to feature grid coords
            feat_x = int(round(sx / src_w * (fw - 1)))
            feat_y = int(round(sy / src_h * (fh - 1)))
            feat_x = min(max(feat_x, 0), fw - 1)
            feat_y = min(max(feat_y, 0), fh - 1)

            # Extract the feature vector at that grid cell
            target_feat = f_src[:, feat_y, feat_x]  # shape: (D,)

            # Compute cosine similarity between this vector and every target patch
            sim = torch.einsum('c,chw->hw', target_feat, f_trg)  # shape: (fh, fw)

            # Find the best-matching location in the target feature map
            if use_softargmax:
                pred_x_feat, pred_y_feat = softargmax_2d(
                    sim,
                    temperature=softargmax_temp,
                    window_size=softargmax_window,
                )
                pred_x_feat = pred_x_feat.item()
                pred_y_feat = pred_y_feat.item()
            else:
                # Simple argmax: pick the single highest-similarity cell
                flat = sim.argmax().item()
                pred_y_feat = float(flat // fw)
                pred_x_feat = float(flat % fw)

            # Convert predicted feature-grid location back to pixel coordinates
            pred_x = ((pred_x_feat + 0.5) / fw) * trg_w
            pred_y = ((pred_y_feat + 0.5) / fh) * trg_h

            # Euclidean distance between prediction and ground truth (in pixels)
            dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)

            total_kps += 1
            image_total_kps += 1

            # A prediction is 'correct' if distance <= threshold x bbox_size
            for t in thresholds:
                if dist <= (t * norm_factor):
                    correct_kps[t] += 1
                    image_correct[t] += 1

        per_image_results.append(
            {
                'category': category,
                'src_image': src_name,
                'trg_image': trg_name,
                'total_kps': image_total_kps,
                **{
                    f'PCK@{t}': (image_correct[t] / image_total_kps * 100.0)
                    if image_total_kps > 0
                    else 0.0
                    for t in thresholds
                },
            }
        )

    # Aggregate results: overall PCK per threshold
    per_keypoint = {
        f'PCK@{t}': (correct_kps[t] / total_kps * 100.0) if total_kps > 0 else 0.0
        for t in thresholds
    }

    results = {
        'model_key': model_key,
        'use_softargmax': bool(use_softargmax),
        'softargmax_temp': float(softargmax_temp),
        'softargmax_window': int(softargmax_window),
        'per_keypoint': per_keypoint,
        'per_image': per_image_results,
        'total_keypoints': int(total_kps),
        'total_image_pairs': int(len(per_image_results)),
    }

    print('\nPer-keypoint PCK:')
    for t in thresholds:
        print(f'  PCK@{t}: {per_keypoint["PCK@" + str(t)]:.2f}%')

    if save_csv:
        os.makedirs(RESULTS_ROOT, exist_ok=True)
        suffix = 'softargmax' if use_softargmax else 'argmax'
        out_csv = os.path.join(RESULTS_ROOT, f'{model_key}-{suffix}_per_image_results.csv')
        pd.DataFrame(per_image_results).to_csv(out_csv, index=False)
        print('Saved per-image CSV:', out_csv)

    return results


# Example usage:
# results = evaluate_cached('dinov2_vitl14', use_softargmax=False, max_pairs=200)
# results = evaluate_cached('dinov2_vitl14', use_softargmax=True, softargmax_temp=0.01, softargmax_window=7, max_pairs=200)


In [ ]:
# ============================================================
# CELL 10 — Run everything and show results
# This is the main execution cell. It does three things:
#   1. Pre-compute and cache features for all 1800 images
#      across all three models (skips images already cached)
#   2. Evaluate each model using argmax and compute PCK scores
#   3. Print a summary table and save results to Drive as CSV
# Expected output: a table comparing DINOv2, DINOv3, and SAM
# ============================================================

precompute_features(list(MODEL_SPECS.keys()))

eval_summaries = []
for model_key in MODEL_SPECS.keys():
    res_argmax = evaluate_cached(
        model_key,
        use_softargmax=False,
        save_csv=True,
    )
    eval_summaries.append(
        {
            "model": model_key,
            "mode": "argmax",
            **res_argmax["per_keypoint"],
            "total_keypoints": res_argmax["total_keypoints"],
            "total_image_pairs": res_argmax["total_image_pairs"],
        }
    )

summary_df = pd.DataFrame(eval_summaries)
os.makedirs(RESULTS_ROOT, exist_ok=True)
summary_path = os.path.join(RESULTS_ROOT, "summary_per_keypoint_pck.csv")
summary_df.to_csv(summary_path, index=False)
print("\nSaved summary:", summary_path)
display(summary_df)

Found 1800 images under /content/data/SPair-71k/JPEGImages
Models: ['dinov2_vitl14', 'dinov3_vitb16', 'sam_vit_l_res1024']


dinov2_vitl14: caching:   0%|          | 0/1800 [00:00<?, ?it/s]

dinov2_vitl14: newly cached 1708 feature files


dinov3_vitb16: caching:   0%|          | 0/1800 [00:00<?, ?it/s]

dinov3_vitb16: newly cached 1800 feature files


...

Evaluating dinov3_vitb16:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 35.77%
  PCK@0.1: 53.91%
  PCK@0.15: 63.57%
  PCK@0.2: 69.50%
Saved per-image CSV: /content/drive/MyDrive/results/dinov3_vitb16-argmax_per_image_results.csv


Evaluating sam_vit_l_res1024:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 15.42%
  PCK@0.1: 24.62%
  PCK@0.15: 32.25%
  PCK@0.2: 38.97%
Saved per-image CSV: /content/drive/MyDrive/results/sam_vit_l_res1024-argmax_per_image_results.csv

Saved summary: /content/drive/MyDrive/results/summary_per_keypoint_pck.csv


,model,mode,PCK@0.05,PCK@0.1,PCK@0.15,PCK@0.2,total_keypoints,total_image_pairs
0,dinov2_vitl14,argmax,37.162621,54.777647,64.395209,71.096368,88328,12234
1,dinov3_vitb16,argmax,35.766688,53.913821,63.565347,69.502310,88328,12234
2,sam_vit_l_res1024,argmax,15.424328,24.624128,32.247985,38.967258,88328,12234


# Stage 2 — Finetune last backbone layers

Unfreeze the last **N** transformer blocks of each backbone and fine-tune with keypoint
supervision from SPair-71k (InfoNCE / contrastive loss).

- **N = 0** — frozen baseline (no training, pure evaluation of pretrained weights)
- **N ∈ {1, 2, 4}** — unfreeze last N blocks + final norm / neck

All 12 runs (3 backbones × 4 values of N) are defined in the next cell.
Results accumulate in `summary_finetune.csv`.

In [ ]:
# All 12 runs (3 backbones × 4 N values). Nothing to edit between runs.
FINETUNE_RUNS = []

_BASE = {
    "lr":               5e-6,
    "epochs":           2,
    "max_train_pairs":  10000,
    "temperature":      0.07,
    "grad_accum_steps": 4,
    "use_amp":          True,
    "checkpoint_dir":   os.path.join(MYDRIVE, "checkpoints", "finetuned"),
}

for _model_key in ["dinov2_vitl14", "dinov3_vitb16", "sam_vit_l_res1024"]:
    for _N in [0, 1, 2, 4]:
        FINETUNE_RUNS.append({"model_key": _model_key, "unfreeze_n": _N, **_BASE})

print(f"Total runs planned: {len(FINETUNE_RUNS)}")
for r in FINETUNE_RUNS:
    print(f"  {r['model_key']}  N={r['unfreeze_n']}")

In [ ]:
import random
import numpy as np
from torch.utils.data import DataLoader


# ── 1. Reload pretrained weights ──────────────────────────────────────────────
def reload_pretrained(model_key):
    global sam, sam_predictor
    if model_key == "dinov2_vitl14":
        dinov2.load_state_dict(torch.load(DINOV2_WEIGHTS, map_location="cpu"))
        dinov2.to(device)
    elif model_key == "dinov3_vitb16":
        dinov3.load_state_dict(torch.load(DINOV3_WEIGHTS, map_location="cpu"))
        dinov3.to(device)
    elif model_key == "sam_vit_l_res1024":
        sam = sam_model_registry["vit_l"](checkpoint=SAM_WEIGHTS_B)
        sam_predictor = SamPredictor(sam)
        sam.to(device)
        MODEL_SPECS["sam_vit_l_res1024"]["model"] = sam_predictor
    print(f"  Reloaded pretrained weights: {model_key}")


# ── 2. Freeze all / unfreeze last N blocks ────────────────────────────────────
def _get_backbone(model_key):
    if model_key == "dinov2_vitl14":
        return dinov2
    elif model_key == "dinov3_vitb16":
        return dinov3
    else:  # sam_vit_l_res1024
        return sam_predictor.model.image_encoder


def set_unfreeze(model_key, N):
    backbone = _get_backbone(model_key)
    for p in backbone.parameters():
        p.requires_grad = False
    if N > 0:
        for block in backbone.blocks[-N:]:
            for p in block.parameters():
                p.requires_grad = True
        for attr in ("norm", "neck"):
            if hasattr(backbone, attr):
                for p in getattr(backbone, attr).parameters():
                    p.requires_grad = True
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in backbone.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


# ── 3. Training-mode feature extraction (gradients enabled) ───────────────────
def extract_ft(model_key, img_tensor):
    if model_key in ("dinov2_vitl14", "dinov3_vitb16"):
        model = dinov2 if model_key == "dinov2_vitl14" else dinov3
        out   = model.forward_features(img_tensor)
        if isinstance(out, dict):
            patch_tokens = out.get("x_norm_patchtokens", out.get("x_norm_patch_tokens"))
        else:
            patch_tokens = out
        b, n, d = patch_tokens.shape
        g = int(math.isqrt(n))
        return patch_tokens.permute(0, 2, 1).reshape(b, d, g, g)
    else:  # SAM
        return sam_predictor.model.image_encoder(img_tensor)


# ── 4. SPair training dataset ─────────────────────────────────────────────────
def _sam_preprocess(pil_img, res):
    arr  = cv2.resize(np.array(pil_img), (res, res)).astype(np.float32)
    t    = torch.from_numpy(arr).permute(2, 0, 1)
    mean = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1)
    std  = torch.tensor([58.395,  57.12,  57.375]).view(-1, 1, 1)
    return (t - mean) / std


class SPairDataset(torch.utils.data.Dataset):
    def __init__(self, root, model_key, max_pairs=10000):
        self.image_dir = os.path.join(root, "JPEGImages")
        self.model_key = model_key
        pair_ann_root  = os.path.join(root, "PairAnnotation", "trn")
        all_files = [
            os.path.join(r, fn)
            for r, _, fs in os.walk(pair_ann_root)
            for fn in fs if fn.endswith(".json")
        ]
        random.seed(42)
        random.shuffle(all_files)
        self.pair_files = all_files[:max_pairs]
        print(f"SPairDataset [{model_key}]: {len(self.pair_files)} training pairs")

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        with open(self.pair_files[idx]) as f:
            ann = json.load(f)
        cat     = ann["category"]
        src_pil = Image.open(os.path.join(self.image_dir, cat, ann["src_imname"])).convert("RGB")
        trg_pil = Image.open(os.path.join(self.image_dir, cat, ann["trg_imname"])).convert("RGB")
        sw, sh  = src_pil.size
        tw, th  = trg_pil.size

        spec = MODEL_SPECS[self.model_key]
        if spec["kind"] == "dino":
            t          = _DINO_TRANSFORMS[self.model_key]
            src_tensor = t(src_pil)
            trg_tensor = t(trg_pil)
        else:
            res        = spec["sam_res"]
            src_tensor = _sam_preprocess(src_pil, res)
            trg_tensor = _sam_preprocess(trg_pil, res)

        src_kps, trg_kps = ann["src_kps"], ann["trg_kps"]
        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        kps = []
        for i in kp_indices:
            ps, pt = src_kps[i], trg_kps[i]
            if ps is not None and pt is not None:
                kps.append({"src": (ps[0] / sw, ps[1] / sh),
                             "trg": (pt[0] / tw, pt[1] / th)})
        return {"src": src_tensor, "trg": trg_tensor, "kps": kps}


def _collate_fn(batch):
    return {
        "src": torch.stack([b["src"] for b in batch]),
        "trg": torch.stack([b["trg"] for b in batch]),
        "kps": [b["kps"] for b in batch],
    }


# ── 5. Contrastive correspondence loss (InfoNCE) ──────────────────────────────
def correspondence_loss(f_src, f_trg, kps_list, temperature=0.07):
    """
    f_src, f_trg : (B, D, H, W)
    kps_list     : list of B lists, each [{src: (x_n, y_n), trg: (x_n, y_n)}, ...]
    """
    B, D, H, W = f_src.shape
    device  = f_src.device
    losses  = []
    for b in range(B):
        for kp in kps_list[b]:
            sx = min(max(int(round(kp["src"][0] * (W - 1))), 0), W - 1)
            sy = min(max(int(round(kp["src"][1] * (H - 1))), 0), H - 1)
            tx = min(max(int(round(kp["trg"][0] * (W - 1))), 0), W - 1)
            ty = min(max(int(round(kp["trg"][1] * (H - 1))), 0), H - 1)
            src_d   = F.normalize(f_src[b, :, sy, sx], dim=0)
            trg_all = F.normalize(f_trg[b].view(D, -1), dim=0)
            sim     = torch.einsum("d,dn->n", src_d, trg_all) / temperature
            losses.append(F.cross_entropy(
                sim.unsqueeze(0),
                torch.tensor([ty * W + tx], device=device)))
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=device)


print("Stage 2 helpers defined.")

In [ ]:
def run_finetune_sweep(runs):
    import shutil
    os.makedirs(runs[0]["checkpoint_dir"], exist_ok=True)
    csv_path = os.path.join(RESULTS_ROOT, "summary_finetune.csv")

    for cfg in runs:
        model_key = cfg["model_key"]
        N         = cfg["unfreeze_n"]
        print(f"\n{'='*60}")
        print(f"  {model_key}  |  unfreeze last N={N}")
        print(f"{'='*60}")

        reload_pretrained(model_key)
        set_unfreeze(model_key, N)

        if N > 0:
            dataset  = SPairDataset(SPAIR_ROOT, model_key, cfg["max_train_pairs"])
            loader   = DataLoader(dataset, batch_size=1, shuffle=True,
                                  num_workers=0, collate_fn=_collate_fn)
            backbone = _get_backbone(model_key)
            opt      = torch.optim.AdamW(
                           filter(lambda p: p.requires_grad, backbone.parameters()),
                           lr=cfg["lr"])
            use_amp  = cfg["use_amp"] and device == "cuda"
            scaler   = torch.amp.GradScaler("cuda", enabled=use_amp)

            backbone.train()
            for epoch in range(cfg["epochs"]):
                opt.zero_grad()
                running = 0.0
                for step, batch in enumerate(tqdm(loader, desc=f"ep{epoch+1}/{cfg['epochs']}")):
                    src = batch["src"].to(device)
                    trg = batch["trg"].to(device)
                    kps = batch["kps"]
                    with torch.autocast("cuda", enabled=use_amp):
                        loss = correspondence_loss(
                            extract_ft(model_key, src),
                            extract_ft(model_key, trg),
                            kps,
                            cfg["temperature"],
                        ) / cfg["grad_accum_steps"]
                    scaler.scale(loss).backward()
                    running += loss.item() * cfg["grad_accum_steps"]
                    if (step + 1) % cfg["grad_accum_steps"] == 0:
                        scaler.step(opt)
                        scaler.update()
                        opt.zero_grad()
                print(f"  epoch {epoch + 1} avg loss: {running / len(loader):.4f}")

            ckpt = os.path.join(cfg["checkpoint_dir"], f"{model_key}_last{N}.pth")
            torch.save(backbone.state_dict(), ckpt)
            print("Checkpoint saved:", ckpt)

        # Clear feature cache so evaluate_cached recomputes from current weights.
        cache_dir = os.path.join(FEATURE_ROOT, model_key)
        if os.path.isdir(cache_dir):
            shutil.rmtree(cache_dir)

        _get_backbone(model_key).eval()
        results = evaluate_cached(model_key, use_softargmax=False, save_csv=False)
        pck     = results["per_keypoint"]

        row = {
            "model_key":       model_key,
            "N":               N,
            "lr":              cfg["lr"],
            "epochs":          cfg["epochs"] if N > 0 else 0,
            "max_train_pairs": cfg["max_train_pairs"] if N > 0 else 0,
            **pck,
        }
        write_header = not os.path.exists(csv_path)
        pd.DataFrame([row]).to_csv(csv_path, mode="a", header=write_header, index=False)
        print(pd.DataFrame([row]).to_string(index=False))

    print(f"\nAll {len(runs)} runs complete. Results saved to: {csv_path}")


run_finetune_sweep(FINETUNE_RUNS)